In [ ]:
import pandas as pd         
print("It works!")

It works!


In [13]:
df = pd.read_csv("../01_Raw_Data/call_center_quality_data.csv")
df.head()

,agent_id,agent_name,month,day,tenure_months,call_type,aht_sec,csat_score,qa_audit_score,resolution_status
0,AGT-019,Kavya Gupta,July-2026,18,28,Inbound,268.4,5.0,88.0,Resolved
1,AGT-001,Arjun Jenkins,February-2026,14,16,INBOUND,283.1,4.3,97.1,Unresolved
2,AGT-018,Sarah Iyer,February-2026,1,23,Inbound,258.1,5.0,95.6,Resolved
3,AGT-007,Rahul Iyer,May-2026,2,27,Inbound,244.0,NaN,100.0,Resolved
4,AGT-012,Kavya Jenkins,March-2026,5,17,Inbound,202.8,4.6,90.4,Resolved


In [14]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3915 entries, 0 to 3914
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   agent_id           3915 non-null   str    
 1   agent_name         3915 non-null   str    
 2   month              3915 non-null   str    
 3   day                3915 non-null   int64  
 4   tenure_months      3915 non-null   int64  
 5   call_type          3915 non-null   str    
 6   aht_sec            3798 non-null   float64
 7   csat_score         3798 non-null   float64
 8   qa_audit_score     3798 non-null   float64
 9   resolution_status  3915 non-null   str    
dtypes: float64(3), int64(2), str(5)
memory usage: 306.0 KB


In [15]:
# Check total missing values per column
print("Missing values per column:")
print(df.isnull().sum())

# Check for duplicate rows
print("\nNumber of duplicate rows:", df.duplicated().sum())

# Check unique values in call_type (to spot inconsistent casing)
print("\nUnique values in call_type: ")
print(df['call_type'].unique())

# Check for stray whitespace in agent_name
print("\nSample agent_name values with possible whitespace:")
print(df['agent_name'].apply(lambda x: x != x.strip()).sum(), "rows have leading/trailing whitespace")

# Check for negative outliers in aht_sec
print("\nRows with negative aht_sec:")
print(df[df['aht_sec'] < 0][['agent_id', 'aht_sec']])

Missing values per column:
agent_id               0
agent_name             0
month                  0
day                    0
tenure_months          0
call_type              0
aht_sec              117
csat_score           117
qa_audit_score       117
resolution_status      0
dtype: int64

Number of duplicate rows: 13

Unique values in call_type: 
<StringArray>
['Inbound', 'INBOUND']
Length: 2, dtype: str

Sample agent_name values with possible whitespace:
157 rows have leading/trailing whitespace

Rows with negative aht_sec:
     agent_id  aht_sec
891   AGT-009   -243.1
2370  AGT-016   -308.8
2542  AGT-013   -288.7
3506  AGT-001   -279.8
3540  AGT-022   -201.2


In [7]:
print("Minimum aht_sec value::", df['aht_sec'].min())
print("Maximum aht_sec value:", df['aht_sec'].max())
print(df['aht_sec'].describe())

Minimum aht_sec value:: 132.3
Maximum aht_sec value: 422.6
count    3797.000000
mean      276.909955
std        43.486324
min       132.300000
25%       246.900000
50%       277.300000
75%       305.600000
max       422.600000
Name: aht_sec, dtype: float64


In [16]:
# Fix missing values: fill with the median (middle value) of each column
# Median is safer than mean here because it's less affected by outliers (like our negative aht_sec values)
df['aht_sec'] = df['aht_sec'].fillna(df['aht_sec'].median())
df['csat_score'] = df['csat_score'].fillna(df['csat_score'].median())
df['qa_audit_score'] = df['qa_audit_score'].fillna(df['qa_audit_score'].median())

# Confirm no missing values remain
print(df.isnull().sum())

agent_id             0
agent_name           0
month                0
day                  0
tenure_months        0
call_type            0
aht_sec              0
csat_score           0
qa_audit_score       0
resolution_status    0
dtype: int64


In [17]:
# Fix negative outliers: convert negative aht_sec values back to positive
# We know these were originally valid, realistic numbers before we artificially made them negative
df['aht_sec'] = df['aht_sec'].abs()

# Confirm no negative values remain
print("Rows with negative aht_sec:", (df['aht_sec'] < 0).sum())
print("Minimum aht_sec value:", df['aht_sec'].min())

Rows with negative aht_sec: 0
Minimum aht_sec value: 120.0


In [18]:
# Remove exact duplicate rows, keeping the first occurence 
df = df.drop_duplicates()

# Confirm duplicates are gone
print("Number of duplicate rows remaining:", df.duplicated().sum())
print("Total rows now:", len(df))

Number of duplicate rows remaining: 0
Total rows now: 3902


In [19]:
# Standardize call_type casing - convert everything to title case (e.g', "Inbound")
df['call_type'] = df['call_type'].str.strip().str.title()

# Confirm only one unique value remains
print(df['call_type'].unique())

<StringArray>
['Inbound']
Length: 1, dtype: str


In [20]:
# Remove leading/trailing whitespace from agent_name
df['agent_name'] = df['agent_name'].str.strip()

# Confirm no whitespace issues remain
whitespace_count= df['agent_name'].apply(lambda x: x != x.strip()).sum()
print("Rows with whitespace issues remaining:", whitespace_count)

Rows with whitespace issues remaining: 0


In [21]:
# Derived column 1: convert AHT from seconds to minutes (easier to read/interpret)
df['aht_minutes'] = round(df['aht_sec'] /60, 2)

# Derived column 2: create a performance tier based on QA audit score
def get_performance_tier(score):
    if score >= 90:
        return 'Excellent'
    elif score >= 75:
        return 'Good'
    elif score >= 60:
        return 'Average'
    else:
        return 'Needs Improvement'

df['performance_tier'] = df['qa_audit_score'].apply(get_performance_tier)

# Check the new columns
print(df[['aht_sec', 'aht_minutes', 'qa_audit_score', 'performance_tier']].head())

   aht_sec  aht_minutes  qa_audit_score performance_tier
0    268.4         4.47            88.0             Good
1    283.1         4.72            97.1        Excellent
2    258.1         4.30            95.6        Excellent
3    244.0         4.07           100.0        Excellent
4    202.8         3.38            90.4        Excellent


In [22]:
# Save the cleaned dataset to a new CSV file
df.to_csv("../02_Working_Files/cleaned_call_center_data.csv", index=False)
print("Cleaned data saved successfully!")
print("Final shape:", df.shape)

Cleaned data saved successfully!
Final shape: (3902, 12)
